# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is accessed via a Croissant schema at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed (uncomment and run the next line if necessary)
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print the main metadata
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
List available record sets, fields, and their `@id` identifiers.

In [ ]:
# List all record sets in the dataset
print("Available record sets and their fields:\n")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"RecordSet name: {rs.name} | @id: {rs.id}")
    for field in rs.fields:
        print(f"  Field: {field.name} | @id: {field.id} | dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load each record set as a DataFrame for exploration. Use the `@id` values from the overview above for precision.

*(Example code uses all record sets in this Croissant package. You may select individual ones based on your analysis goals.)*

In [ ]:
# Extract and display all record sets into pandas DataFrames, keyed by their @id
dataframes = {}

# Store a mapping of record set @id to the object for field lookup
record_set_map = {rs.id: rs for rs in dataset.metadata.record_sets}

for rs in dataset.metadata.record_sets:
    rs_id = rs.id
    print(f"Loading records for RecordSet @id: {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records into DataFrame with columns:", df.columns.tolist())
        display(df.head())
    else:
        print(f"No records found for RecordSet @id: {rs_id}.")
    print("---\n")
# For this dataset, the main analytic record set is usually the primary tabular dataset. If the dataset only has one record set, you may reference it below by its @id.

# For demonstration, select the first non-empty record set
main_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rs_id
        break

if main_record_set_id is not None:
    print(f"Using main RecordSet @id for further exploration: {main_record_set_id}")
    print("Columns: ", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No suitable tabular record set found.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data cleaning and transformations, such as filtering, normalization, and grouping.

**Note:** Choose fields for numeric and group analysis by their `@id`. Some common fields from clinical datasets may include age, sex, interval (days between cancers), etc.

In [ ]:
import numpy as np

df = dataframes[main_record_set_id]
# Identify numeric fields by inspecting the record set
rs_obj = record_set_map[main_record_set_id]
numeric_fields = [f.id for f in rs_obj.fields if f.data_type in ["Float", "Number", "Integer"]]
print("Numeric field ids:", numeric_fields)

# Pick the first numeric field as an example
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field for demonstration: {numeric_field_id}")
else:
    print("No numeric fields available in the main record set.")

if numeric_fields:
    # Filtering: e.g., select patients age > 50 if field represents age
    threshold = None
    # Set a reasonable threshold, e.g., 50 if it's age, or 0 or 10 for intervals/continuous indicators
    # Try to guess field kind by @id and field name
    field_name = [f.name for f in rs_obj.fields if f.id == numeric_field_id][0]
    # Heuristic: If 'age' in the name (case-insensitive), set threshold=50
    if 'age' in field_name.lower():
        threshold = 50
    else:
        threshold = 10

    print(f"Filtering records where field {numeric_field_id} ({field_name}) > {threshold}.")

    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records (count={len(filtered_df)}) where {field_name} > {threshold}:")
    display(filtered_df.head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"Normalized {field_name} for filtered entries:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Look for a category/group field (e.g., sex, anatomical_site, etc)
    group_candidate_fields = [f for f in rs_obj.fields if f.data_type == "Text"]
    if group_candidate_fields:
        # Default to first text field
        group_field_id = group_candidate_fields[0].id
        group_field_name = group_candidate_fields[0].name
        print(f"Grouping by field: {group_field_id} ({group_field_name})")
        grouped_df = filtered_df[[group_field_id, numeric_field_id]].groupby(group_field_id).mean()
        print(f"Grouped mean {field_name} by {group_field_name}:")
        display(grouped_df)
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Create basic visualizations to explore field distributions or relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    # Histogram
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10)
    plt.title(f"Distribution of {field_name}")
    plt.xlabel(field_name)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field
    if group_candidate_fields:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{field_name} by {group_field_name}")
        plt.xlabel(group_field_name)
        plt.ylabel(field_name)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Skipping visualizations: no numeric field available.")

## 6. Conclusion
In this notebook, you loaded and explored the FAIR² clinical oncology dataset via its Croissant schema and the `mlcroissant` library. By referencing fields and record sets via their `@id`, the notebook demonstrated best practices for standardized, schema-driven data exploration. This framework enables robust data processing, normalization, and visualization suitable for downstream machine learning or statistical analysis tasks.